In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix

In [2]:
train = pd.read_parquet(
    "../data/processed/train.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

print("Train:", train.shape)
print("Validation:", validation.shape)

Train: (1928949, 5)
Validation: (413345, 5)


# Decide what constitutes a collaborative interaction

In [3]:
user_item = (
    train
    .groupby(["user_id", "item_id"])
    ["interaction_strength"]
    .sum()
    .reset_index()
)

print(user_item.shape)
print(user_item.head())

(1496539, 3)
   user_id  item_id  interaction_strength
0        3   385090                     1
1        5    61396                     1
2        7   139394                     1
3        7   164941                     1
4        7   226353                     1


# Check sparsity

In [4]:
num_users = user_item["user_id"].nunique()
num_items = user_item["item_id"].nunique()

print("Unique users:", num_users)
print("Unique items:", num_items)
print("User-item interactions:", len(user_item))

Unique users: 978906
Unique items: 200974
User-item interactions: 1496539


In [5]:
total_possible = num_users * num_items

print(
    "Sparsity:",
    1 - len(user_item) / total_possible
)

Sparsity: 0.9999923931093674


# Create user/item indices

In [6]:
user_ids = user_item["user_id"].unique()
item_ids = user_item["item_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

item_to_idx = {
    item_id: idx
    for idx, item_id in enumerate(item_ids)
}

# Create the sparse user-item matrix

In [7]:
rows = user_item["user_id"].map(
    user_to_idx
).to_numpy()

cols = user_item["item_id"].map(
    item_to_idx
).to_numpy()

values = user_item[
    "interaction_strength"
].astype(np.float32).to_numpy()

In [8]:
user_item_matrix = csr_matrix(
    (
        values,
        (rows, cols)
    ),
    shape=(
        len(user_ids),
        len(item_ids)
    ),
    dtype=np.float32
)

In [9]:
print(
    "Matrix shape:",
    user_item_matrix.shape
)

print(
    "Non-zero values:",
    user_item_matrix.nnz
)

Matrix shape: (978906, 200974)
Non-zero values: 1496539


# Identify warm validation users

In [10]:
train_users = set(
    train["user_id"].unique()
)

validation_users = set(
    validation["user_id"].unique()
)

warm_validation_users = (
    validation_users & train_users
)

print(
    "Warm validation users:",
    len(warm_validation_users)
)

Warm validation users: 18965


# Restrict collaborative data to warm users

In [11]:
cf_train = user_item[
    user_item["user_id"].isin(
        warm_validation_users
    )
].copy()

print("CF interactions:", len(cf_train))
print(
    "CF users:",
    cf_train["user_id"].nunique()
)

print(
    "CF items:",
    cf_train["item_id"].nunique()
)

CF interactions: 74023
CF users: 18965
CF items: 40654


# Build the warm user-item matrix

In [12]:
cf_user_ids = (
    cf_train["user_id"]
    .unique()
)

cf_item_ids = (
    cf_train["item_id"]
    .unique()
)

cf_user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(cf_user_ids)
}

cf_item_to_idx = {
    item_id: idx
    for idx, item_id in enumerate(cf_item_ids)
}

In [13]:
cf_rows = cf_train["user_id"].map(
    cf_user_to_idx
).to_numpy()

cf_cols = cf_train["item_id"].map(
    cf_item_to_idx
).to_numpy()

cf_values = cf_train[
    "interaction_strength"
].astype(np.float32).to_numpy()

In [14]:
cf_matrix = csr_matrix(
    (
        cf_values,
        (cf_rows, cf_cols)
    ),
    shape=(
        len(cf_user_ids),
        len(cf_item_ids)
    ),
    dtype=np.float32
)

In [15]:
print(
    "CF matrix:",
    cf_matrix.shape
)

print(
    "Non-zero:",
    cf_matrix.nnz
)

CF matrix: (18965, 40654)
Non-zero: 74023


# Normalize the user vectors

In [16]:
from sklearn.preprocessing import normalize

cf_user_matrix = normalize(
    cf_matrix,
    norm="l2",
    axis=1
)

# Test similarity for ONE user

In [17]:
sample_user = cf_user_ids[0]

sample_idx = cf_user_to_idx[
    sample_user
]

user_vector = cf_user_matrix[
    sample_idx
]

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarities = cosine_similarity(
    user_vector,
    cf_user_matrix
).ravel()

In [19]:
user_similarities[sample_idx] = -1

In [20]:
top_user_indices = np.argsort(
    user_similarities
)[::-1][:10]

In [21]:
similar_users = pd.DataFrame({
    "user_id": cf_user_ids[
        top_user_indices
    ],
    "similarity": user_similarities[
        top_user_indices
    ]
})

print(
    "Target user:",
    sample_user
)

print(similar_users)

Target user: 39
   user_id  similarity
0  1407573         0.0
1   471195         0.0
2   471352         0.0
3   471391         0.0
4   471434         0.0
5   471493         0.0
6   471516         0.0
7   471644         0.0
8   471647         0.0
9   471661         0.0


In [22]:
item_user_matrix = cf_matrix.T

print("Item-user matrix:", item_user_matrix.shape)

Item-user matrix: (40654, 18965)


In [23]:
item_user_matrix_norm = normalize(
    item_user_matrix,
    norm="l2",
    axis=1
)

print(
    "Non-zero values:",
    item_user_matrix_norm.nnz
)

Non-zero values: 74023


In [24]:
sample_item = cf_item_ids[0]

sample_item_idx = cf_item_to_idx[
    sample_item
]

print("Sample item:", sample_item)

Sample item: 76740


In [25]:
item_similarities = cosine_similarity(
    item_user_matrix_norm[sample_item_idx],
    item_user_matrix_norm
).ravel()

item_similarities[sample_item_idx] = -1

top_item_indices = np.argsort(
    item_similarities
)[::-1][:10]

In [26]:
similar_items_cf = pd.DataFrame({
    "item_id": cf_item_ids[
        top_item_indices
    ],
    "similarity": item_similarities[
        top_item_indices
    ]
})

print(
    "Target item:",
    sample_item
)

print(similar_items_cf)

Target item: 76740
   item_id  similarity
0   235724         0.0
1   188390         0.0
2     6962         0.0
3    42317         0.0
4   359990         0.0
5   212052         0.0
6   152627         0.0
7   323801         0.0
8   265513         0.0
9   435960         0.0
